In [6]:
!pip install -qU langchain-groq langchain-core langgraph

In [7]:
import os
import json
import re
from typing import TypedDict, List, Dict, Any
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# 0. API Key setup karna
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# 1. Workflow State Define Karein
# TypedDict specify karta hai ki ek node se doosre node mein kaunsa data flow karega
class CareerState(TypedDict, total=False):
    answers: Dict[str, Any]
    userProfile: str
    recommendedPaths: List[Dict[str, Any]]
    selectedPath: str
    roadmap: List[Dict[str, Any]]
    resumeHtml: str

def get_llm():
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )

In [8]:
# Node 1: Profile Evaluator
def evaluate_profile(state: CareerState):
    print("-> 🧠 Node 1: Evaluating Profile...")
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert career profiler. Summarize the user's skills, core interests, and strengths in a concise paragraph."),
        ("user", "User's assessment answers: {answers}")
    ])
    chain = prompt | llm | StrOutputParser()
    profile = chain.invoke({"answers": json.dumps(state.get("answers", {}))})

    return {"userProfile": profile}

In [9]:
# Node 2: Path Recommender
def recommend_paths(state: CareerState):
    print("-> 🛤️ Node 2: Recommending Career Paths...")
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Based on the user's profile, recommend **exactly 3** job paths. Format strictly as a JSON array with keys: 'title', 'fitExplanation', 'skillsToDevelop' (array of strings). Return only raw JSON without markdown formatting."),
        ("user", "User Profile: {userProfile}")
    ])
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"userProfile": state.get("userProfile")})

    # JSON ko clean karna
    cleaned = re.sub(r'```json\n?|```\n?', '', response).strip()
    paths = json.loads(cleaned)

    # Aage ke steps ke liye top path select karna (ensure it's safe even if less than 3 are returned)
    top_path = paths[0].get("title", "Software Developer") if paths else "Software Developer"

    return {"recommendedPaths": paths, "selectedPath": top_path}

In [10]:
# Node 3: Roadmap Generator
def generate_roadmap(state: CareerState):
    selected_path = state.get("selectedPath")
    print(f"-> 🗺️ Node 3: Generating Roadmap for {selected_path}...")
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a 4-step upskilling roadmap for the job path '{pathTitle}'. Format as a JSON array of objects with keys: 'stepNumber', 'title', 'description', 'estimatedTime' (e.g., '2 weeks', '1 month'), 'resourceLinks' (array of URLs). Return only raw JSON."),
        ("user", "Generate the roadmap.")
    ])
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"pathTitle": selected_path})

    cleaned = re.sub(r'```json\n?|```\n?', '', response).strip()
    roadmap = json.loads(cleaned)

    return {"roadmap": roadmap}

In [11]:
# Node 4: Resume Generator
def generate_resume(state: CareerState):
    selected_path = state.get("selectedPath")
    print(f"-> 📄 Node 4: Generating Resume for {selected_path}...")
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a professional HTML resume tailored for the job role: {jobRole}. Use the user profile: {profile}. Format as valid HTML inside a <div> using Tailwind classes (e.g., text-xl, font-bold). Output ONLY raw HTML."),
        ("user", "Generate the resume.")
    ])
    chain = prompt | llm | StrOutputParser()
    html = chain.invoke({
        "jobRole": selected_path,
        "profile": state.get("userProfile")
    })

    return {"resumeHtml": html}

In [12]:
# --- BUILD THE WORKFLOW GRAPH ---
def build_graph():
    workflow = StateGraph(CareerState)

    # Nodes add karein
    workflow.add_node("evaluateProfile", evaluate_profile)
    workflow.add_node("recommendPaths", recommend_paths)
    workflow.add_node("generateRoadmap", generate_roadmap)
    workflow.add_node("generateResume", generate_resume)

    # Flow / Edges connect karein
    workflow.add_edge(START, "evaluateProfile")
    workflow.add_edge("evaluateProfile", "recommendPaths")
    workflow.add_edge("recommendPaths", "generateRoadmap")
    workflow.add_edge("generateRoadmap", "generateResume")
    workflow.add_edge("generateResume", END)

    return workflow.compile()

In [16]:
import re
from typing import TypedDict, List, Dict, Any
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Markdown

# --- RUN THE AUTOMATION ---
def main():
    applet_workflow = build_graph()

    # Get user input for assessment data
    print("Please provide your career assessment answers:")
    degree = input("Your degree (e.g., B.Tech Computer Science (Tier-3)): ")
    experience = input("Your experience (e.g., Fresher, 2 years in software): ")
    programming_languages_str = input("Programming languages you know (comma-separated, e.g., Python, JavaScript, SQL): ")
    interests_str = input("Your interests (comma-separated, e.g., Data Analysis, Building web apps): ")
    work_style = input("Your preferred work style (e.g., Loves to work independently and solve logic puzzles): ")

    user_answers = {
        "degree": degree,
        "experience": experience,
        "programmingLanguages": [lang.strip() for lang in programming_languages_str.split(',')] if programming_languages_str else [],
        "interests": [interest.strip() for interest in interests_str.split(',')] if interests_str else [],
        "workStyle": work_style
    }

    print("🚀 Starting Career Automation Workflow...\n")

    try:
        # LangGraph invoke karna
        final_state = applet_workflow.invoke({"answers": user_answers})

        print("\n✅ WORKFLOW COMPLETED SUCCESSFULLY!\n")

        # Display Generated Profile
        profile_text = final_state.get("userProfile", "No profile generated.")
        display(Markdown(f"## Generated Profile\n\n{profile_text}\n"))

        # Display All Recommended Paths
        recommended_paths = final_state.get("recommendedPaths", [])
        if recommended_paths:
            display(Markdown("## Recommended Career Paths\n"))
            for i, path in enumerate(recommended_paths):
                display(Markdown(f"### Path {i+1}: {path.get('title')}\n\n**Explanation:** {path.get('fitExplanation')}\n\n**Skills to Develop:** {', '.join(path.get('skillsToDevelop', []))}\n"))
        else:
            display(Markdown("## Recommended Career Paths\nNo recommended paths generated.\n"))

        # Display Roadmap (First Step Snippet with Estimated Time)
        roadmap = final_state.get("roadmap", [])
        if roadmap:
            display(Markdown("## Roadmap (First Step)\n"))
            first_step = roadmap[0]
            display(Markdown(f"### Step {first_step.get('stepNumber')}: {first_step.get('title')}\n\n**Description:** {first_step.get('description')}\n\n**Estimated Time:** {first_step.get('estimatedTime')}\n\n**Resource Links:** {', '.join(first_step.get('resourceLinks', []))}\n"))
        else:
            display(Markdown("## Roadmap (First Step)\nNo roadmap steps generated.\n"))

        # Display Resume HTML (Snippet)
        html_snippet = final_state.get("resumeHtml", "")
        display(Markdown("## Resume HTML Snippet\n"))
        if html_snippet:
            display(Markdown(f"```html\n{html_snippet[:500]}...\n```"))
        else:
            display(Markdown("No resume HTML generated.\n"))

    except Exception as e:
        print(f"❌ Workflow failed: {e}")

In [14]:
# Script run karein
if __name__ == "__main__":
    main()

Please provide your career assessment answers:
Your degree (e.g., B.Tech Computer Science (Tier-3)): cwrf
Your experience (e.g., Fresher, 2 years in software): th
Programming languages you know (comma-separated, e.g., Python, JavaScript, SQL): ujy
Your interests (comma-separated, e.g., Data Analysis, Building web apps): yukj
Your preferred work style (e.g., Loves to work independently and solve logic puzzles): yj
🚀 Starting Career Automation Workflow...

-> 🧠 Node 1: Evaluating Profile...
-> 🛤️ Node 2: Recommending Career Paths...
-> 🗺️ Node 3: Generating Roadmap for Innovation Consultant...
-> 📄 Node 4: Generating Resume for Innovation Consultant...

✅ WORKFLOW COMPLETED SUCCESSFULLY!

--- GENERATED PROFILE ---
Based on the user's assessment, it appears they have a unique blend of skills and interests. With a degree in "cwrf" and experience in "th", they may possess a strong foundation in a specialized field. Their proficiency in programming languages such as "ujy" suggests they have 